# 03 — Modelagem

**Issues #7–#10** · Milestone M3

- **#7** Classificação: LR, Random Forest, SVM
- **#8** Regressão: Linear, Random Forest (`money_diff`)
- **#9** KNN (Aula 5)
- **#10** ROC, matriz de confusão, feature importance, `model_results.json`

> Notebook **autocontido** — carrega `rounds_start.csv` ou gera a partir do CSV bruto.

In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
N_JOBS = 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

try:
    import kagglehub
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
SVM_MAX_SAMPLES = 6_000
REGRESSION_EXCLUDE = {"money_diff", "money_ratio", "ct_money", "t_money"}

DATA_RAW = Path("/content/data/raw")
DATA_PROCESSED = Path("/content/data/processed")
FIGURES = Path("/content/reports/figures")
METRICS = Path("/content/reports/metrics")
for p in (DATA_RAW, DATA_PROCESSED, FIGURES, METRICS):
    p.mkdir(parents=True, exist_ok=True)

CSV_RAW = DATA_RAW / "csgo_round_snapshots.csv"
CSV_PROCESSED = DATA_PROCESSED / "rounds_start.csv"
DATASET = "christianlillelund/csgo-round-winner-classification"

In [ ]:
MAP_EXCLUDE = {"de_cache"}
RIFLE_COLS_CT = [
    "ct_weapon_ak47", "ct_weapon_m4a4", "ct_weapon_m4a1s",
    "ct_weapon_galilar", "ct_weapon_famas", "ct_weapon_sg553", "ct_weapon_aug",
]
RIFLE_COLS_T = [c.replace("ct_", "t_") for c in RIFLE_COLS_CT]
UTIL_COLS_CT = [
    "ct_grenade_hegrenade", "ct_grenade_flashbang", "ct_grenade_smokegrenade",
    "ct_grenade_incendiarygrenade", "ct_grenade_molotovgrenade", "ct_grenade_decoygrenade",
]
UTIL_COLS_T = [c.replace("ct_", "t_") for c in UTIL_COLS_CT]


BUY_TIER_ORDER = ["force", "eco", "mixed", "half", "full"]


def buy_tier(team_money: float) -> str:
    """Fase de compra a partir do dinheiro total do time (soma dos 5 jogadores).

    Proxy: média por jogador (team_money / 5). Faixas CS:GO competitivo:
    force < $1.700, eco < $2.000, half $2.800–$3.799, full >= $4.700, mixed nas transições.
    Saldo mínimo pós-compra (half) e loss bonus não estão no dataset.
    """
    m = team_money / 5
    if m >= 4700:
        return "full"
    if 2800 <= m < 3800:
        return "half"
    if m < 1700:
        return "force"
    if m < 2000:
        return "eco"
    return "mixed"


def filter_round_start(df: pd.DataFrame) -> pd.DataFrame:
    mask = (
        (df["time_left"] >= 150)
        & (df["ct_players_alive"] == 5)
        & (df["t_players_alive"] == 5)
        & (~df["map"].isin(MAP_EXCLUDE))
    )
    return df.loc[mask].copy()


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    new_cols = pd.DataFrame(
        {
            "money_diff": out["ct_money"] - out["t_money"],
            "health_diff": out["ct_health"] - out["t_health"],
            "armor_diff": out["ct_armor"] - out["t_armor"],
            "players_alive_diff": out["ct_players_alive"] - out["t_players_alive"],
            "money_ratio": out["ct_money"] / out["t_money"].replace(0, 1),
            "ct_has_awp": (out["ct_weapon_awp"] > 0).astype(int),
            "t_has_awp": (out["t_weapon_awp"] > 0).astype(int),
            "ct_rifle_count": out[RIFLE_COLS_CT].sum(axis=1),
            "t_rifle_count": out[RIFLE_COLS_T].sum(axis=1),
            "ct_util_count": out[UTIL_COLS_CT].sum(axis=1),
            "t_util_count": out[UTIL_COLS_T].sum(axis=1),
            "ct_buy_tier": out["ct_money"].map(buy_tier),
            "t_buy_tier": out["t_money"].map(buy_tier),
            "target_cls": (out["round_winner"] == "T").astype(int),
        }
    )
    return pd.concat([out, new_cols], axis=1)


def get_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_base = [
        "time_left", "ct_score", "t_score", "ct_health", "t_health",
        "ct_armor", "t_armor", "ct_money", "t_money", "ct_helmets", "t_helmets",
        "ct_defuse_kits", "ct_players_alive", "t_players_alive",
    ]
    derived = [
        "money_diff", "health_diff", "armor_diff", "players_alive_diff",
        "money_ratio", "ct_has_awp", "t_has_awp", "ct_rifle_count",
        "t_rifle_count", "ct_util_count", "t_util_count",
    ]
    weapon_cols = [c for c in df.columns if c.startswith(("ct_weapon_", "t_weapon_"))]
    grenade_cols = [c for c in df.columns if c.startswith(("ct_grenade_", "t_grenade_"))]
    numeric = numeric_base + derived + weapon_cols + grenade_cols + ["bomb_planted"]
    numeric = [c for c in numeric if c in df.columns]
    categorical = [c for c in ["map", "ct_buy_tier", "t_buy_tier"] if c in df.columns]
    return numeric, categorical


def make_preprocessor(df: pd.DataFrame, exclude: set[str] | None = None) -> ColumnTransformer:
    numeric, categorical = get_feature_columns(df)
    if exclude:
        numeric = [c for c in numeric if c not in exclude]
        categorical = [c for c in categorical if c not in exclude]
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ],
        remainder="drop",
    )


def ensure_raw_csv() -> Path:
    if CSV_RAW.exists():
        return CSV_RAW
    cache = Path(kagglehub.dataset_download(DATASET))
    for src in cache.rglob("*.csv"):
        shutil.copy2(src, DATA_RAW / src.name)
    return CSV_RAW


def load_processed_df() -> pd.DataFrame:
    if CSV_PROCESSED.exists():
        print("Carregando processado:", CSV_PROCESSED)
        df = pd.read_csv(CSV_PROCESSED)
    else:
        print("Gerando base processada...")
        df = add_derived_features(filter_round_start(pd.read_csv(ensure_raw_csv())))
        df.to_csv(CSV_PROCESSED, index=False)
    # Recalcula buy tiers (CSV em cache pode ter faixas antigas)
    df["ct_buy_tier"] = df["ct_money"].map(buy_tier)
    df["t_buy_tier"] = df["t_money"].map(buy_tier)
    return df


df = load_processed_df()
print(f"Base modelagem: {len(df):,} linhas | T={df['target_cls'].mean():.1%}")

In [ ]:
numeric, categorical = get_feature_columns(df)
feature_cols = numeric + categorical

X_cls = df[feature_cols]
y_cls = df["target_cls"]
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cls
)

reg_cols = [c for c in feature_cols if c not in REGRESSION_EXCLUDE]
X_reg = df[reg_cols]
y_reg = df["money_diff"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

preprocessor_cls = make_preprocessor(df)
preprocessor_reg = make_preprocessor(df, exclude=REGRESSION_EXCLUDE)
print("Treino classificação:", X_train.shape, "| regressão:", X_train_r.shape)

## 1. Classificação — LR, RF, SVM, KNN (issues #7 e #9)

In [ ]:
def subsample_stratified(X, y, max_samples: int):
    if len(X) <= max_samples:
        return X, y
    X_sub, _, y_sub, _ = train_test_split(
        X, y, train_size=max_samples, stratify=y, random_state=RANDOM_STATE
    )
    return X_sub, y_sub


def roc_auc_score_estimator(estimator, X_test, y_test) -> float:
    if hasattr(estimator, "predict_proba"):
        scores = estimator.predict_proba(X_test)[:, 1]
    else:
        scores = estimator.decision_function(X_test)
    return float(roc_auc_score(y_test, scores))


X_svm, y_svm = subsample_stratified(X_train, y_train, SVM_MAX_SAMPLES)

classifier_configs = [
    (
        "logistic_regression",
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight="balanced"),
        {"clf__C": [0.1, 1.0, 10.0]},
        X_train, y_train,
    ),
    (
        "random_forest",
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=N_JOBS),
        {"clf__max_depth": [12, 20], "clf__min_samples_leaf": [1, 5]},
        X_train, y_train,
    ),
    (
        "svm_rbf",
        SVC(kernel="rbf", random_state=RANDOM_STATE, class_weight="balanced"),
        {"clf__C": [0.5, 1.0], "clf__gamma": ["scale"]},
        X_svm, y_svm,
    ),
    (
        "knn",
        KNeighborsClassifier(),
        {"clf__n_neighbors": [5, 11, 21]},
        X_train, y_train,
    ),
]

cls_results = {}
fitted_cls = {}
best_name, best_f1 = None, -1.0

for name, estimator, param_grid, X_fit, y_fit in classifier_configs:
    print(f"Treinando {name} ({len(X_fit):,} amostras)...")
    t0 = time.perf_counter()
    pipe = Pipeline([("prep", preprocessor_cls), ("clf", estimator)])
    grid = GridSearchCV(pipe, param_grid, cv=3, scoring="f1", n_jobs=-1, refit=True)
    grid.fit(X_fit, y_fit)
    model = grid.best_estimator_
    fitted_cls[name] = model
    y_pred = model.predict(X_test)
    elapsed = time.perf_counter() - t0
    metrics = {
        "best_params": grid.best_params_,
        "cv_best_f1": round(float(grid.best_score_), 4),
        "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "f1": round(float(f1_score(y_test, y_pred)), 4),
        "roc_auc": round(roc_auc_score_estimator(model, X_test, y_test), 4),
        "train_samples": len(X_fit),
        "elapsed_seconds": round(elapsed, 1),
    }
    cls_results[name] = metrics
    if metrics["f1"] > best_f1:
        best_f1, best_name = metrics["f1"], name
    print(f"  F1={metrics['f1']:.4f} AUC={metrics['roc_auc']:.4f} ({elapsed:.0f}s)")

cls_results["best_model"] = best_name
pd.DataFrame(cls_results).T.drop(index="best_model", errors="ignore")

## 2. Regressão — `money_diff` (issue #8)

In [ ]:
regressor_configs = [
    ("linear_regression", LinearRegression(), {}, False),
    (
        "random_forest_regressor",
        RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=N_JOBS),
        {"reg__max_depth": [12, 20], "reg__min_samples_leaf": [1, 5]},
        True,
    ),
]

reg_results = {}
fitted_reg = {}
best_reg_name, best_rmse = None, float("inf")

for name, estimator, param_grid, use_grid in regressor_configs:
    print(f"Treinando {name}...")
    t0 = time.perf_counter()
    pipe = Pipeline([("prep", preprocessor_reg), ("reg", estimator)])
    if use_grid:
        grid = GridSearchCV(pipe, param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
        grid.fit(X_train_r, y_train_r)
        model = grid.best_estimator_
        cv_rmse = round(float(-grid.best_score_), 2)
        best_params = grid.best_params_
    else:
        pipe.fit(X_train_r, y_train_r)
        model = pipe
        cv_rmse = round(
            float(-cross_val_score(pipe, X_train_r, y_train_r, cv=3, scoring="neg_root_mean_squared_error", n_jobs=N_JOBS).mean()),
            2,
        )
        best_params = {}
    fitted_reg[name] = model
    y_pred = model.predict(X_test_r)
    elapsed = time.perf_counter() - t0
    rmse = float(np.sqrt(mean_squared_error(y_test_r, y_pred)))
    metrics = {
        "best_params": best_params,
        "cv_rmse": cv_rmse,
        "mae": round(float(mean_absolute_error(y_test_r, y_pred)), 2),
        "rmse": round(rmse, 2),
        "r2": round(float(r2_score(y_test_r, y_pred)), 4),
        "elapsed_seconds": round(elapsed, 1),
    }
    reg_results[name] = metrics
    if rmse < best_rmse:
        best_rmse, best_reg_name = rmse, name
    print(f"  RMSE={metrics['rmse']:.2f} R²={metrics['r2']:.4f} ({elapsed:.0f}s)")

reg_results["best_model"] = best_reg_name
pd.DataFrame(reg_results).T.drop(index="best_model", errors="ignore")

## 3. Gráficos e métricas (issue #10)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, model in fitted_cls.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name.replace("_", " "))
ax.set_title("Curvas ROC — classificação")
plt.show()
fig.savefig(FIGURES / "10_roc_classificacao.png")
plt.close(fig)

best_cls = fitted_cls[best_name]
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(best_cls, X_test, y_test, ax=ax, cmap="Blues")
ax.set_title(f"Matriz de confusão — {best_name}")
plt.show()
fig.savefig(FIGURES / "11_matriz_confusao.png")
plt.close(fig)

if "random_forest" in fitted_cls:
    rf = fitted_cls["random_forest"]
    prep = rf.named_steps["prep"]
    clf = rf.named_steps["clf"]
    names = prep.get_feature_names_out()
    imp = clf.feature_importances_
    idx = np.argsort(imp)[-15:]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(names[idx], imp[idx])
    ax.set_title("Top 15 — importância (RF classificação)")
    plt.tight_layout()
    plt.show()
    fig.savefig(FIGURES / "12_importancia_classificacao.png")
    plt.close(fig)

best_reg = fitted_reg[best_reg_name]
y_pred_r = best_reg.predict(X_test_r)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test_r, y_pred_r, alpha=0.2, s=8)
lims = [min(y_test_r.min(), y_pred_r.min()), max(y_test_r.max(), y_pred_r.max())]
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlabel("money_diff real")
ax.set_ylabel("money_diff previsto")
ax.set_title(f"Regressão — {best_reg_name}")
plt.show()
fig.savefig(FIGURES / "13_regressao_real_vs_previsto.png")
plt.close(fig)

if "random_forest_regressor" in fitted_reg:
    rf_reg = fitted_reg["random_forest_regressor"]
    names = rf_reg.named_steps["prep"].get_feature_names_out()
    imp = rf_reg.named_steps["reg"].feature_importances_
    idx = np.argsort(imp)[-15:]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(names[idx], imp[idx])
    ax.set_title("Top 15 — importância (RF regressão)")
    plt.tight_layout()
    plt.show()
    fig.savefig(FIGURES / "14_importancia_regressao.png")
    plt.close(fig)

print("Gráficos salvos em:", FIGURES)

In [ ]:
summary = {
    "rows_used": len(df),
    "svm_max_train_samples": SVM_MAX_SAMPLES,
    "classification": cls_results,
    "regression": reg_results,
}
out = METRICS / "model_results.json"
out.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("Métricas:", out)
print(f"Melhor classificador: {best_name} (F1={best_f1:.4f})")
print(f"Melhor regressão: {best_reg_name} (RMSE={best_rmse:.2f})")